# 9.2 DQN의 안정화: 타겟 네트워크 — 실습 노트북

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml2/chapter09_2_target_network.ipynb)

책 본문: [9.2 DQN의 안정화 장치와 실습](https://smhanlab.com/book-ml/kor/ml2/chapter09.html)

이 노트북은 9.2절의 내용을 그대로 실행합니다.
(1) "움직이는 목표"를 손으로 확인하고, (2) 값반복(value iteration)으로 참값을
구하고, (3) 타겟 네트워크 **없음(A) / 있음(B)** 두 팔(arm)을 1차원 랜덤워크
환경에서 돌려서 책에 든 학습곡선 그림을 재생산합니다.
경험 재현은 9.3절의 주제이므로 여기선 의도적으로 쓰지 않습니다 —
두 팔의 차이는 **타겟 네트워크 하나**에만 두는 통제된 대조 실험입니다.

In [1]:
import matplotlib
matplotlib.use("Agg")
from matplotlib import font_manager
import matplotlib.pyplot as plt
kr = [f.name for f in font_manager.fontManager.ttflist if "Noto Sans CJK KR" in f.name]
if kr: plt.rcParams["font.sans-serif"] = [kr[0]]
plt.rcParams["axes.unicode_minus"] = False
IMG = "/home/smhan/book-ml/kor/src/images"

import random
import numpy as np
import torch
import torch.nn as nn

## 1. "움직이는 목표"를 손으로 확인하기

장난감: \(Q(s,a;\theta) = \theta\), \(r = 1\), \(\gamma = 0.99\).
목표값은 \(r + \gamma\theta\)인데, 타겟 네트워크가 없으면 \(\theta\)가
매 스텝 갱신될 때마다 목표 자체도 바뀝니다. \(\theta\)가
\(0 \to 0.5 \to 0.9\)로 변한다고 하면:

In [2]:
gamma = 0.99
print("타겟 네트워크가 없을 때 (목표 = 1 + 0.99*θ — θ 갱신마다 바뀜):")
for th in [0.0, 0.5, 0.9]:
    print(f"  θ = {th:<4} → 목표 = {1 + gamma*th:.3f}")
print()
print("타겟 네트워크가 있을 때 (θ⁻ = 0 고정):")
for th in [0.5, 0.9]:
    print(f"  θ = {th:<4} → 목표 = {1 + gamma*0.0:.3f}  (그대로 고정)")

타겟 네트워크가 없을 때 (목표 = 1 + 0.99*θ — θ 갱신마다 바뀜):
  θ = 0.0  → 목표 = 1.000
  θ = 0.5  → 목표 = 1.495
  θ = 0.9  → 목표 = 1.891

타겟 네트워크가 있을 때 (θ⁻ = 0 고정):
  θ = 0.5  → 목표 = 1.000  (그대로 고정)
  θ = 0.9  → 목표 = 1.000  (그대로 고정)


## 2. 오차 수명 \(1/(1-\gamma)\)

미래의 오차는 \(\gamma^k\)만큼 할인되어 현재로 전달되므로, "신뢰할 수 있는
미래"의 길이는 사실상 \(1/(1-\gamma)\)스텝이고, 오차의 반감기는
\(\log 0.5 / \log \gamma\)스텝입니다 — 책의 표와 일치하는지 확인합니다:

In [3]:
for g in [0.9, 0.99, 0.999]:
    life = 1.0 / (1.0 - g)
    half = np.log(0.5) / np.log(g)
    print(f"γ = {g:<5} 오차 수명 1/(1-γ) = {life:>6,.0f} 스텝,   반감기 ≈ {half:>5,.0f} 스텝")

γ = 0.9   오차 수명 1/(1-γ) =     10 스텝,   반감기 ≈     7 스텝
γ = 0.99  오차 수명 1/(1-γ) =    100 스텝,   반감기 ≈    69 스텝
γ = 0.999 오차 수명 1/(1-γ) =  1,000 스텝,   반감기 ≈   693 스텝


## 3. 환경: 1차원 랜덤워크 + 값반복으로 참값

상태 \(x \in \{0, \dots, 12\}\), 행동 = 왼쪽(0)/오른쪽(1).
행동대로 한 칸 이동할 확률 0.9, 반대 방향으로 미끄러질 확률 0.1.
매 스텝 \(-1\) (시간 비용), \(x=12\) 목표 (+10, 종료), \(x=0\) 위험
(\(-10\), 종료), \(\gamma = 0.99\).
참값 \(Q^{\*}\)은 벨만 방정식의 고정점으로, 5,000회 반복하면 완전히
수렴합니다. 판정 기준이 되는 탐침 상태는 \(x = 8\)입니다:

In [4]:
X_MAX, GAMMA = 12, 0.99
STEP_R, GOAL_R, HIT_R, DRIFT = -1.0, 10.0, -10.0, 0.9

def trans(x, a):
    """상태 x에서 행동 a의 (집계된) 전이: [(x2, r, done, prob)]"""
    delta = +1 if a == 1 else -1          # 행동대로 갈 확률 DRIFT
    cands = [(min(X_MAX, x + delta), DRIFT), (max(0, x - delta), 1.0 - DRIFT)]
    agg = {}
    for x2, p in cands:
        agg[x2] = agg.get(x2, 0.0) + p
    out = []
    for x2, p in agg.items():
        done = (x2 == 0) or (x2 == X_MAX)
        r = STEP_R + (GOAL_R if x2 == X_MAX else HIT_R if x2 == 0 else 0.0)
        out.append((x2, r, done, p))
    return out

class RandomWalk:
    """1차원 랜덤워크 환경. reset()은 1~11 사이 임의 상태에서 시작한다."""
    def reset(self):
        return random.randint(1, X_MAX - 1)

    def step(self, x, a):
        delta = +1 if a == 1 else -1
        x2 = min(X_MAX, x + delta) if random.random() < DRIFT else max(0, x - delta)
        done = (x2 == 0) or (x2 == X_MAX)
        r = STEP_R + (GOAL_R if x2 == X_MAX else HIT_R if x2 == 0 else 0.0)
        return x2, r, done

def true_values(n_iter=5000):
    """값반복: 벨만 최적 방정식의 고정점 Q*."""
    Q = np.zeros((X_MAX + 1, 2))
    for _ in range(n_iter):
        nQ = Q.copy()
        for x in range(1, X_MAX):            # 비터미널 상태만 갱신
            for a in range(2):
                val = 0.0
                for x2, r, done, p in trans(x, a):
                    val += p * (r + (0.0 if done else GAMMA * np.max(Q[x2])))
                nQ[x, a] = val
        Q = nQ
    return Q

Qtrue = true_values()
print(f"Q*(8, 왼쪽)  = {Qtrue[8,0]:.4f}   (책: 2.642)")
print(f"Q*(8, 오른쪽) = {Qtrue[8,1]:.4f}   (책: 4.720)")
print(f"탐침 상태 x=8: 오른쪽(목표 방향)이 최적이고, 이점 ≈ {Qtrue[8,1] - Qtrue[8,0]:.3f} (책: 약 2.08)")

Q*(8, 왼쪽)  = 2.6415   (책: 2.642)
Q*(8, 오른쪽) = 4.7197   (책: 4.720)
탐침 상태 x=8: 오른쪽(목표 방향)이 최적이고, 이점 ≈ 2.078 (책: 약 2.08)


## 4. 두 팔 실험: 타겟 네트워크 있음 vs 없음

- **(A) 타겟 없음** — 목표값을 *현재* \(\theta\)로 계산 (9.1절의 식 그대로).
  그래디언트 계산도 안에 들어가므로 "목표도 함께 당겨지는" 경로까지 생김.
- **(B) 타겟 있음** — 하드 업데이트, \(C = 200\)스텝마다 통째로 복사.

두 팔은 이 외 전부 동일: 시드 0, 1→16→16→2, Adam \(lr = 10^{-2}\),
\(\varepsilon = 0.1\) 고정, 25,000스텝, 온-폴리시·재현 없음.
함수만 정의해 둡니다:

In [5]:
class QNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(1, 16), nn.ReLU(),
                                 nn.Linear(16, 16), nn.ReLU(),
                                 nn.Linear(16, 2))

    def forward(self, x):
        return self.net(x)

def run_arm(use_target, seed=0, steps=25000, lr=1e-2, sync=200, eps=0.1, max_ep=40):
    torch.manual_seed(seed); random.seed(seed); np.random.seed(seed)
    net = QNet()
    tnet = None
    if use_target:
        tnet = QNet(); tnet.load_state_dict(net.state_dict())   # 처음엔 둘이 동일
    opt = torch.optim.Adam(net.parameters(), lr=lr)             # Q_net만! 타겟은 학습 안 함
    env = RandomWalk()
    probe_x = 8
    rec_steps, rec_qr, rec_ql, rec_loss = [], [], [], []
    step_id = 0
    while step_id < steps:
        x = env.reset()
        ep = 0
        done = False
        while not done and ep < max_ep:
            s1 = torch.tensor([[x / X_MAX]])
            with torch.no_grad():
                q = net(s1)
            a = random.randrange(2) if random.random() < eps else int(q.argmax().item())
            x2, r, done = env.step(x, a)
            s2 = torch.tensor([[x2 / X_MAX]])
            mask = 0.0 if done else 1.0
            if use_target:
                with torch.no_grad():
                    tgt = r + GAMMA * tnet(s2).max(1).values * mask   # 고정된 목표
            else:
                tgt = r + GAMMA * net(s2).max(1).values * mask        # 움직이는 목표(그래프 안)
            pred = net(s1).gather(1, torch.tensor([[a]])).squeeze()
            loss = (pred - tgt) ** 2
            opt.zero_grad(); loss.backward(); opt.step()
            step_id += 1
            if step_id % 25 == 0:                                    # 25스텝마다 기록
                with torch.no_grad():
                    qv = net(torch.tensor([[probe_x / X_MAX]])).numpy()[0]
                rec_steps.append(step_id); rec_qr.append(qv[1]); rec_ql.append(qv[0])
                rec_loss.append(loss.item())
            if use_target and step_id % sync == 0:                   # 하드 업데이트
                tnet.load_state_dict(net.state_dict())
            x, ep = x2, ep + 1
            if done or ep >= max_ep:
                break
    return np.array(rec_steps), np.array(rec_qr), np.array(rec_ql), np.array(rec_loss)

두 팔을 순서대로 돌립니다 (총 25,000 × 2 스텝 — 30초 내외 소요):

In [6]:
stA, qrA, qlA, loA = run_arm(use_target=False)
stB, qrB, qlB, loB = run_arm(use_target=True)

def report(tag, qr, lo):
    final = qr[-1]
    pct = (final - Qtrue[8, 1]) / Qtrue[8, 1] * 100
    print(f"{tag}: 최종 Q(8, 오른쪽) = {final:.3f}  (참값 대비 {pct:+.0f}%)   "
          f"TD 손실(마지막 100샘플) = {lo[-100:].mean():.2f}")

report("A. 타겟 없음        ", qrA, loA)
report("B. 타겟 있음 (C=200)", qrB, loB)
print(f"   참값 Q*(8, 오른쪽) = {Qtrue[8,1]:.3f}")

A. 타겟 없음        : 최종 Q(8, 오른쪽) = 6.012  (참값 대비 +27%)   TD 손실(마지막 100샘플) = 0.46
B. 타겟 있음 (C=200): 최종 Q(8, 오른쪽) = 4.961  (참값 대비 +5%)   TD 손실(마지막 100샘플) = 1.32
   참값 Q*(8, 오른쪽) = 4.720


## 5. 학습곡선 (책 그림: ch09_2_target_net_effect.svg)

(a) 탐침 상태 \(x=8\)의 \(Q(\text{오른쪽})\) — 타겟이 없으면(주황)
참값 4.72보다 *높은* 약 6.0에 안정되고, 타겟이 있으면(파랑) 참값에 가까워진다.
(b) TD 손실(로그 스케일) — 타겟이 없는 쪽이 오히려 *낮게* 수렴한다.
읽을 점 세 가지: **(1)** 움직이는 목표는 "떨림"이 아니라 "값을 잘못된 곳에
세운다" (과대추정); **(2)** 과대추정은 \(\max\)의 구조적 편향 +
부트스트래핑 증폭 — 9.3절 Double DQN이 겨냥하는 것; **(3)** *낮은 TD 손실 ≠
올바른 Q* — (B)의 잔여 손실(≈1.3)은 "C스텝 낡은 목표"의 대가이며, 학습
진행의 척도는 손실이 아니라 **평가 리턴**이다:

In [7]:
def moving_avg(y, w=40):
    return np.convolve(y, np.ones(w) / w, mode="same")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.plot(stA / 1000, moving_avg(qrA), color="#e6550d", lw=2.2, label="A: 타겟 없음 (움직이는 목표)")
ax1.plot(stB / 1000, moving_avg(qrB), color="#3182bd", lw=2.2, label="B: 타겟 있음 (C=200)")
ax1.axhline(Qtrue[8, 1], color="black", ls="--", lw=1.5, label=f"참값 Q* = {Qtrue[8,1]:.2f}")
ax1.set_xlabel("학습 스텝 (×10³)")
ax1.set_ylabel("Q(8, 오른쪽)")
ax1.set_title("(a) 탐침 상태 x=8의 Q값")
ax1.legend(fontsize=9); ax1.grid(alpha=0.3)

ax2.plot(stA / 1000, np.maximum(moving_avg(loA), 1e-8), color="#e6550d", lw=2.2, label="A: 타겟 없음")
ax2.plot(stB / 1000, np.maximum(moving_avg(loB), 1e-8), color="#3182bd", lw=2.2, label="B: 타겟 있음")
ax2.set_yscale("log")
ax2.set_xlabel("학습 스텝 (×10³)")
ax2.set_ylabel("TD 손실 (로그)")
ax2.set_title("(b) TD 손실")
ax2.legend(fontsize=9); ax2.grid(alpha=0.3, which="both")

fig.tight_layout()
fig.savefig(IMG + "/ch09_2_target_net_effect.svg", bbox_inches="tight")
plt.show()
print(f"그림 저장 → {IMG + '/ch09_2_target_net_effect.svg'}")

그림 저장 → /home/smhan/book-ml/kor/src/images/ch09_2_target_net_effect.svg


## 6. 정리 — 그리고 다음으로 (9.3)

- **움직이는 목표는 발산이 아니라 "잘못된 값에의 안정화"** — 훈련 로그(손실)는
  멀쩡한데 Q값이 참값보다 27% 높은 곳에 정착한다.
- **타겟 네트워크는 \(C\)스텝만 목표를 동결**해서 "고정된 정답 최적화"로
  만들어 준다. 대가는 목표의 \(C\)스텝 낡음(잔여 TD 손실)이다.
- 과대추정의 *근원*인 \(\max\) 편향을 직접 없애는 **Double DQN**과,
  상관된 데이터를 깨는 **경험 재현**이 다음 절(9.3)에서 같은 학습 루프에
  더해진다. CartPole에서 두 장치가 함께 작동하는 학습곡선을 보게 됩니다.